# Case Study: Clinical Trial (Binary GAMM)

**Duration:** 18 min | **Level:** ⭐⭐⭐

Analyze binary outcomes with nested random effects (patients within clinics).

In [ ]:
import sys
sys.path.append('../')
from utils import load_clinical_trial_data
import numpy as np
from aurora.models.gamm import fit_gamm

df = load_clinical_trial_data()
print(f'Loaded {len(df)} clinical observations')
print(f'Patients: {df["patient"].nunique()}, Clinics: {df["clinic"].nunique()}')
print(df.head())

In [ ]:
# Binary GAMM: outcome ~ treatment + time + (1|clinic) + (1|patient)
n = len(df)
n_clinics = df['clinic'].nunique()
n_patients = df['patient'].nunique()

X = np.column_stack([
    np.ones(n),
    df['treatment'],
    df['time'],
    df['treatment'] * df['time']  # Interaction
])
y = df['outcome'].values

# Random effects: clinic + patient
Z = np.zeros((n, n_clinics + n_patients))
for i, (clinic, patient) in enumerate(zip(df['clinic'], df['patient'])):
    Z[i, clinic] = 1
    Z[i, n_clinics + patient] = 1

result = fit_gamm(X, y, Z, family='binomial', method='pql')
print(f'\nBinary GAMM Results:')
for name, coef in zip(['Intercept', 'Treatment', 'Time', 'Treat×Time'], result.fixed_effects):
    or_val = np.exp(coef)
    print(f'{name:<15} OR={or_val:.2f}')

In [ ]:
# Success rates over time
import matplotlib.pyplot as plt

success_rate = df.groupby(['treatment', 'time'])['outcome'].mean().unstack()

plt.figure(figsize=(10, 6))
for treat in [0, 1]:
    label = 'Control' if treat == 0 else 'Treatment'
    plt.plot(success_rate.columns, success_rate.loc[treat], 'o-', label=label, linewidth=2, markersize=8)

plt.xlabel('Time (weeks)')
plt.ylabel('Success Rate')
plt.title('Clinical Trial: Treatment Effect Over Time')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

✅ Binary GAMM for longitudinal binary outcomes  
✅ Nested random effects (clinic + patient)  
✅ Treatment effect increases over time  
✅ Accounts for clustering within clinics  

## Key Findings

- Treatment group shows improved outcomes
- Effect amplifies with time (positive interaction)
- Significant inter-clinic variability
- Patient-level random effects capture repeated measures

---
**Author:** Lucy E. Arias  
**Date:** 2025-11-04